# Explore Elexon wind data

Fetch a small sample of FUELHH (WIND) actuals and WINDFOR forecast revisions, inspect their units/columns, then prove the D-1 cutoff selection.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from pipeline.ingest import API_BASE
from pipeline.transform import select_decision_time_forecast

START, END = '2026-01-01T00:00:00Z', '2026-01-04T00:00:00Z'

In [ ]:
def fetch(name, **params):
    response = requests.get(f'{API_BASE}/{name}', params={**params, 'format': 'json'}, timeout=60)
    response.raise_for_status()
    payload = response.json()
    return pd.DataFrame(payload.get('data', payload) if isinstance(payload, dict) else payload)

actuals = fetch('FUELHH', **{'from': START, 'to': END, 'fuelType': 'WIND'})
forecasts = fetch('WINDFOR', **{'from': START, 'to': END})
actuals.head(), forecasts.head()

In [ ]:
print('Actual columns:', actuals.columns.tolist())
print('Forecast columns:', forecasts.columns.tolist())
display(actuals.describe(include='all'))
display(forecasts.describe(include='all'))
# Elexon generation values are interpreted by this pipeline as MW. Verify against API metadata/docs before changing units.

In [ ]:
selected = select_decision_time_forecast(forecasts)
selected.head(10)
# For every target_time, publish_time must be <= cutoff_time (D-1 23:30 UTC).
assert (selected['publish_time'] <= selected['cutoff_time']).all()
selected.assign(lag=selected['cutoff_time'] - selected['publish_time']).head()